<span style="font-size: 5em">🏠</span>

# __Airbnb Multi-City Analysis Agent__
## Build Agentic AI for Real-World Data Workflows

<div style="background: linear-gradient(135deg, #FF5A5F 0%, #00A699 100%); padding: 20px; border-radius: 10px; color: white; margin: 20px 0; border-left: 4px solid #484848;">
    <h3 style="margin: 0; color: white;">🎯 What You'll Build</h3>
    <p>An agent that <strong>automatically cleans data</strong> based on your preferences, then <strong>autonomously decides</strong> what analytics to generate — and applies this across <strong>multiple cities</strong>.</p>
</div>
---


**The key innovations:**
1. Ask preferences ONCE, apply to ALL cities
2. LLM autonomously decides what analytics to create based on the data
3. LLM decides the most impactful visualization for your audience


---
## The Challenge

Imagine you get handed **multiple CSV files** — same structure, different slices of data. Each has dozens of columns, messy values, outliers, and missing data. You need to:

1. **Pick** which columns are actually relevant
2. **Clean** the data consistently across ALL files
3. **Generate** meaningful insights
4. **Create** visualizations for a presentation

This is a common real-world scenario: multiple regions, time periods, departments — same schema, same mess, multiplied.

> **In this workshop** we'll use Airbnb listings data from several European cities as our test dataset — but the agent we build is designed to work with **any** collection of CSVs that share a structure.

### Why Custom Agents Beat Generic Solutions

| Approach | Problem |
|---|---|
| ❌ **Manual cleaning** | Inconsistent across files, time-consuming, error-prone |
| ❌ **ChatGPT / Claude chat** | Can't maintain state across files — you'd re-explain context every time |
| ❌ **Fully autonomous agents** (AutoGPT etc.) | No human control at key decisions — it guesses what *you* want |
| ❌ **Traditional scripts** | No AI intelligence, breaks on unexpected data, no adaptive analysis |

<div style="background:rgb(30, 48, 32); padding: 15px; border-radius: 8px; border-left: 4px solid #4CAF50; margin: 15px 0;">
    <strong>✅ What we'll build instead:</strong> A task-specific agent that asks for <strong>YOUR</strong> input exactly 3 times — then autonomously processes <strong>ALL</strong> files with your preferences. You stay in control of the decisions that matter; the agent handles the repetitive work.
</div>




---
### A Quick Word on the Tools: LangChain & LangGraph

We're using **LangGraph** (built on top of **LangChain**) to implement this agent. Here's a brief orientation:

- **LangChain** is an open-source framework for building applications powered by LLMs. It gives you standardized interfaces for calling models, managing prompts, and connecting tools — think of it as the plumbing between your code and an LLM.
- **LangGraph** extends LangChain with a **graph-based execution model**: you define your workflow as nodes (processing steps) and edges (transitions), including conditional branching and human-in-the-loop interrupts. It also provides built-in **state management** and **checkpointing**, so the agent can pause, wait for input, and resume exactly where it left off.

<div style="background:rgb(218, 149, 37); padding: 15px; border-radius: 8px; border-left: 4px solid #FF9800; margin: 15px 0;">
    <strong>⚠️ Disclaimer:</strong> This is <strong>not</strong> a LangChain promotion. We picked LangGraph because its graph execution model and built-in interrupt mechanism fit our "ask the human, then run autonomously" pattern really well. You could absolutely build something similar with other agent frameworks.
</div>

**Why LangGraph fits this task:**

| Feature | Why it matters for us |
|---|---|
| 🔀 **Graph structure** | Each processing step is a node — easy to visualize, debug, and extend |
| ⏸️ **Interrupt & resume** | Pause the agent mid-run, collect human input, continue seamlessly |
| 💾 **State checkpointing** | If something crashes, you don't lose progress |
| 🔁 **Conditional edges** | The agent can decide its own path through the workflow |

---
## Part 0: Setup & Environment Check

In [ ]:
# 🔧 SETUP CHECK - Run this cell first!
import sys
print(f"✅ Python version: {sys.version}")

# Check required packages
required = ["langgraph", "langchain", "langchain_openai", "pandas", "matplotlib", "seaborn"]
missing = []

for pkg in required:
    try:
        __import__(pkg.replace("-", "_"))
        print(f"✅ {pkg} installed")
    except ImportError:
        missing.append(pkg)
        print(f"❌ {pkg} missing")

if missing:
    print(f"\n🔴 Install missing packages: pip install {' '.join(missing)}")
else:
    print("\n🟢 All packages ready!")

In [ ]:
# 📦 Import everything we need
import uuid
import json
from pathlib import Path
import pandas as pd
from langgraph.checkpoint.sqlite import SqliteSaver
from IPython.display import Image
from langgraph.types import Command

print("✅ All imports successful!")

---
## Part 1: Explore the Data

We have data from **4 European cities**: Amsterdam, Athens, Barcelona, Berlin

In [ ]:
from dataset_config import DATASET_CONFIG

In [ ]:
# 📂 Discover available data
data_path = DATASET_CONFIG["data_path"]
entity_col = DATASET_CONFIG["entity_column"]

entities = [d.name for d in data_path.iterdir() if d.is_dir()]
print(f"📂 Available {entity_col}: {entities}")

# Check what files exist for each entity
for entity in entities:
    entity_path = data_path / entity
    files = list(entity_path.glob("*.csv*"))
    print(f"\n📁 {entity}:")
    for f in files:
        print(f"   • {f.name}")

---
## Part 2: Define the Multi-City Agent State

We need to track:
- **Multiple datasets** (one per city)
- **Shared preferences** (ask once, apply everywhere)
- **Analysis task** (LLM decides what to do)
- **Results per city**

In [ ]:
# 📋 Import the state definition (see states.py for the full TypedDict)
from states import MultiDatasetAnalysisState

# Quick peek at the state fields:
import inspect, textwrap
source = inspect.getsource(MultiDatasetAnalysisState)

---
## Part 3: Build the Agent Nodes


In [ ]:
# 🔧 Load the processing nodes (this also initializes the LLM connection)
import nodes
# 🎨 Load the visualization nodes
import visualization

---
## Part 4: Build the Graph

In [ ]:
# 🏗️ Build the graph
import build_graph as bg
builder = bg.build_graph()

import sqlite3
SESSION_NAME = DATASET_CONFIG.get("session_name", "analysis")
DB_PATH = f"{SESSION_NAME}_checkpoints.db"
conn = sqlite3.connect(DB_PATH, check_same_thread=False)
checkpointer = SqliteSaver(conn)
agent = builder.compile(checkpointer=checkpointer)

print("✅ Agent compiled!")
print(f"💾 State persists to: {DB_PATH}")
print("\n📊 Flow:")
print("   load_data → detect_issues → ask_preferences (INTERRUPT)")
print("   → clean_all_entities → generate_analysis_prompt → decide_analysis")
print("   → choose_analysis (INTERRUPT) → execute_analysis → decide_visualization")
print("   → choose_visualization (INTERRUPT) → create_visualization → export")

In [ ]:
# 📊 Visualize the graph
display(Image(agent.get_graph().draw_mermaid_png()))

---
## Part 5: Run the Agent!

In [ ]:
# 🚀 Start the agent — edit entities_to_analyze to match your dataset!
initial_state = {
    # Config
    "entity_column": DATASET_CONFIG["entity_column"],

    # Which sub-folders (entities) to analyze
    "entities_to_analyze": ["Athens", "Barcelona", "Berlin"],

    # Internal state — leave as-is
    "entity_datasets": {},
    "detected_issues": None,
    "cleaning_actions": [],
    "user_preferences": [],
    "preferences_collected": False,
    "generated_analysis_prompt": None,
    "generated_viz_prompt": None,
    "analysis_options": None,
    "analysis_task": None,
    "visualization_options": None,
    "visualization_task": None,
    "visualization_figure": None,
    "analysis_results": None,
    "insights": None,
    "summary": None,
}

SESSION_NAME = DATASET_CONFIG.get("session_name", "analysis")
thread_id = f"{SESSION_NAME}_{uuid.uuid4()}"
config = {"configurable": {"thread_id": thread_id}}

print(f"🚀 Starting Multi-Dataset Analysis Agent...")
print(f"   Thread ID: {thread_id}")
print(f"   Entities: {initial_state['entities_to_analyze']}")
print(f"   Entity column: {initial_state['entity_column']}")
print()

result = agent.invoke(initial_state, config)

In [ ]:
# 👀 See what issues were detected
if "__interrupt__" in result:
    print("\n" + "🛑" * 20)
    print("\n⏸️  AGENT PAUSED — WAITING FOR YOUR PREFERENCES")
    print("\n" + "🛑" * 20)
    
    interrupt_data = result["__interrupt__"][0].value
    
    print(f"\n📋 {interrupt_data['message']}")
    print(f"💡 {interrupt_data['note']}\n")
    
    for issue in interrupt_data['issues']:
        severity_emoji = {"high": "🔴", "medium": "🟡", "low": "🟢"}[issue['severity']]
        print(f"┌{'─' * 68}┐")
        print(f"│ {severity_emoji} Issue {issue['id']}: {issue['type'].upper():<50} │")
        print(f"├{'─' * 68}┤")
        print(f"│ Column: {issue['column']:<57} │")
        print(f"│ {issue['description']:<66} │")
        print(f"├{'─' * 68}┤")
        print(f"│ OPTIONS:{'':59} │")
        for num, opt in issue['options'].items():
            print(f"│   {num}. {opt:<61} │")
        print(f"└{'─' * 68}┘")
        print()
else:
    print("\n✅ Agent completed without needing preferences!")

---
## 👤 Provide Your Preferences

Edit the choices below — they'll be applied to **ALL 4 cities**!

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 👤 YOUR CHOICES — Edit these to match your preferences!
# ════════════════════════════════════════════════════════════════════════════

# Look at the issues above and fill in your choices
# These will be applied to ALL datasets consistently!

my_preferences = {
    "preferences": [
        {"issue_id": 0, "choice": "Remove outlier rows"},
        {"issue_id": 1, "choice": "Remove outlier rows"},
        {"issue_id": 2, "choice": "Fill with 0"},
        {"issue_id": 3, "choice": "Remove outlier rows"},
        {"issue_id": 4, "choice": "Remove outlier rows"},
        {"issue_id": 5, "choice": "Fill with median"},
        {"issue_id": 6, "choice": "Keep all values"},
        {"issue_id": 7, "choice": "Keep all values"},
        {"issue_id": 8, "choice": "Fill with most common value"},
        {"issue_id": 9, "choice": "Fill with 0"},
        {"issue_id": 10, "choice": "Keep all values"},

    ]
}

print("📝 Your preferences (will apply to ALL datasets):")
for p in my_preferences["preferences"]:
    print(f"   Issue {p['issue_id']}: {p['choice']}")

In [ ]:
# 🚀 Resume the agent with your preferences!
# It will clean and propose analysis tasks, then PAUSE for you to pick one.

print("\n" + "═" * 70)
print("▶️  RESUMING AGENT WITH YOUR PREFERENCES")
print("═" * 70)

human_response = Command(resume=my_preferences)
result = agent.invoke(human_response, config)

# The agent will pause at the analysis choice interrupt
if "__interrupt__" in result:
    interrupt_data = result["__interrupt__"][0].value
    analysis_options = agent.get_state(config).values.get("analysis_options", [])

    print("\n" + "═" * 70)
    print("⏸️  AGENT PAUSED — PICK AN ANALYSIS")
    print("═" * 70)
    print(f"\n🧠 {interrupt_data.get('message', 'Choose an analysis:')}\n")

    for i, opt in enumerate(analysis_options, 1):
        print(f"  [{i}] {opt['task_name']}")
        print(f"      {opt['task_description']}")
        print(f"      Type: {opt.get('analysis_type', 'N/A')} | Group by: {opt.get('group_by_column', 'None')}\n")

    print("👉 Run the next cell to type your choice!")
else:
    print("\n🎉 Workflow complete!")


In [ ]:
# 👤 Pick an analysis task
print("-" * 50)
your_choice = input("Pick an analysis (1, 2, or 3): ")
print(f"\n▶️  Resuming agent with your choice: Option {your_choice}\n")
print("=" * 70)

result = agent.invoke(Command(resume=str(your_choice)), config)

# The agent will pause at the visualization choice interrupt
if "__interrupt__" in result:
    interrupt_data = result["__interrupt__"][0].value
    viz_options = agent.get_state(config).values.get("visualization_options", [])

    print("\n" + "═" * 70)
    print("⏸️  AGENT PAUSED — PICK A VISUALIZATION")
    print("═" * 70)
    print(f"\n📊 {interrupt_data.get('message', 'Choose a visualization:')}\n")

    for i, opt in enumerate(viz_options, 1):
        print(f"  [{i}] {opt['viz_type'].upper()} — {opt['title']}")
        print(f"      {opt['description']}")
        print(f"      💡 {opt['rationale']}\n")

    print("👉 Run the next cell to type your choice!")
else:
    print("\n🎉 Workflow complete!")

In [ ]:
# 👤 Pick a visualization
print("-" * 50)
your_choice = input("Pick a visualization (1, 2, or 3): ")
print(f"\n▶️  Resuming agent with your choice: Option {your_choice}\n")
print("=" * 70)

final_result = agent.invoke(Command(resume=str(your_choice)), config)

print("\n🎉 Workflow complete!")

---
## 🔬 Inspect Results

In [ ]:
# View detailed results
if final_result.get("analysis_results"):
    print("📊 ANALYSIS RESULTS\n")

    # Entity stats
    print("Entity-Level Statistics:")
    stats_df = pd.DataFrame(final_result["analysis_results"]["entity_stats"])
    display(stats_df.T)

    # Group breakdown (if any)
    if final_result["analysis_results"].get("group_breakdown"):
        print("\nGroup Breakdown:")
        group_df = pd.DataFrame(final_result["analysis_results"]["group_breakdown"])
        display(group_df)
    
    print(f"\nTotal records: {final_result['analysis_results'].get('total_records', 'N/A'):,}")

In [ ]:
# Preview cleaned data for the first entity
if final_result.get("entity_datasets"):
    entity = list(final_result["entity_datasets"].keys())[0]
    entity_data = final_result["entity_datasets"][entity]

    if entity_data.get("cleaned_data"):
        cleaned_df = pd.read_csv(pd.io.common.StringIO(entity_data["cleaned_data"]))
        print(f"📊 Cleaned Data Preview ({entity}):")
        display(cleaned_df.head(10))

        print(f"\n📈 Quick Stats:")
        print(f"   Rows: {len(cleaned_df):,}")
        for col in cleaned_df.select_dtypes(include="number").columns[:3]:
            print(f"   Avg {col}: {cleaned_df[col].mean():.2f}")

---
## 🔁 Reuse Preferences for New Data

**This is the real power!** Once you've set preferences, you can add more cities or process updated data with the same cleaning rules.

In [ ]:
# 💾 Save your preferences for future use
if final_result.get("user_preferences"):
    SESSION_NAME = DATASET_CONFIG.get("session_name", "analysis")
    prefs_file = f"{SESSION_NAME}_cleaning_preferences.json"

    with open(prefs_file, "w") as f:
        json.dump({
            "preferences": final_result["user_preferences"],
            "entity_column": DATASET_CONFIG["entity_column"],
            "created_from_entities": list(final_result["entity_datasets"].keys()),
            "thread_id": thread_id
        }, f, indent=2)

    print(f"💾 Preferences saved to: {prefs_file}")
    print("\n📋 Your saved preferences:")
    for pref in final_result["user_preferences"]:
        print(f"   • {pref['column']}: {pref['choice']}")

    print("\n💡 You can now load these and apply to new entities!")

# 💾 Save cleaned datasets to disk
if final_result.get("entity_datasets"):
    SESSION_NAME = DATASET_CONFIG.get("session_name", "analysis")
    output_dir = Path(f"{SESSION_NAME}_cleaned")
    output_dir.mkdir(exist_ok=True)


    for entity_name, entity_data in final_result["entity_datasets"].items():
        if entity_data.get("cleaned_data"):
            out_path = output_dir / f"{entity_name}_cleaned.csv"
            pd.read_csv(pd.io.common.StringIO(entity_data["cleaned_data"])).to_csv(out_path, index=False)
            print(f"💾 Saved: {out_path}  ({entity_data['row_count']:,} rows)")

---
## 📚 Key Takeaways

### What You Built

1. **Multi-Dataset Processing** — Load and process 4 cities in one workflow
2. **Shared Preferences** — Ask once, apply to all (impossible in Cursor!)
3. **Autonomous Analysis** — LLM decides what analytics to generate
4. **Autonomous Visualization** — LLM decides the most impactful chart for your audience
5. **Persistent State** — Resume anytime, even after kernel restart

### The Workflow

```
Load 4 Cities → Detect Issues → Ask YOUR Preferences → Clean ALL → LLM Decides Task → Analyze → LLM Picks Viz → Create Chart → Report
```

### Why This Matters

| Manual Approach | Your Agent |
|----------------|-------------|
| Clean each city separately | One preference, all cities |
| Decide analysis yourself | LLM suggests best analysis |
| Create charts manually | LLM picks the most impactful viz |
| Repeat for new data | Saved preferences, instant apply |
| No audit trail | Full traceability |

---

<div style="background: linear-gradient(135deg, #FF5A5F 0%, #00A699 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
    <h3 style="color: white;">🎉 Congratulations!</h3>
    <p>You built a <strong>multi-city, preference-aware, autonomous analysis agent</strong> for Airbnb data!</p>
    <p>Your agent asks for preferences ONCE, applies them to ALL cities, lets the LLM decide what to analyze, AND automatically creates the most impactful visualization for your presentation!</p>
</div>